
1. Оценка сложности текста - задача 
2. Классические метрики оценки сложности текста: Статистические метрики, Метрики удобочитаемости, Метрики лексического разнообразия
3. Подходы машинного обучения (регрессия и классификация)
4. Методы на основе эмбеддингов

In [13]:
import os
import pandas as pd
import random
import textstat
import re
from ruts import BasicStats, ReadabilityStats, DiversityStats, MorphStats

In [14]:
# Загрузка текстов 
def load_texts(base_folder, years=('2023','2024')):
    """
    Обходит папки base_folder/IMS<год>, ищет файлы с названием _IMS_<год>[_rus].txt,
    считывает каждый текст и возвращает DataFrame с колонками:
      - Name: имя файла без расширения
      - Year: год (строка '2023' или '2024')
      - Text: полный текст файла
    """
    records = []
    for year in years:
        dir_path = os.path.join(base_folder, f'IMS{year}')
        if not os.path.isdir(dir_path):
            continue
        for fname in os.listdir(dir_path):
            if fname.endswith(f'_IMS_{year}_rus.txt') or fname.endswith(f'_IMS_{year}.txt'):
                path = os.path.join(dir_path, fname)
                with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                    txt = f.read().strip()
                name = os.path.splitext(fname)[0]
                records.append({'Name': name, 'Year': year, 'Text': txt})
    # Превращаем список словарей в DataFrame для дальнейшей работы
    return pd.DataFrame(records)

In [15]:
if __name__ == '__main__':
    base_folder = '/Users/juliak/Downloads/IMS2013-20242' 
    df = load_texts(base_folder)

    # Случайная выборка до 10 текстов
    sample = df.sample(n=min(10, len(df)), random_state=42).reset_index(drop=True)
    
    # Списки для хранения результатов: отдельно для textstat, отдельно для ruts
    ts_list = []
    ruts_list = []
    
    # Проходим по каждому тексту из выборки
    for _, row in sample.iterrows():
        name, year, text = row['Name'], row['Year'], row['Text']

        # textstat
        ts_list.append({
            'Name': name,
            'Year': year,
            'Flesch Reading Ease':         textstat.flesch_reading_ease(text), # «легкость чтения» (0–100, где больше = проще)
            'Flesch-Kincaid Grade':        textstat.flesch_kincaid_grade(text), #уровень школьного класса (более высокое = сложнее)
            'SMOG':                         textstat.smog_index(text), #Опирается на подсчет “сложных” слов (длинные или редкоупотребимые), результат – примерный класс образования
            'Coleman-Liau Index':          textstat.coleman_liau_index(text), #опирается на количество букв на 100 слов и предложений, результат = класс чтения
            'Automated Readability Index': textstat.automated_readability_index(text), #использует символы и слова, результат = класс
            'Gunning Fog':                 textstat.gunning_fog(text), #% «сложных» слов (3+ слога) + длина предложений, результат = класс
            'Dale-Chall Score':            textstat.dale_chall_readability_score(text), #% слов вне списка простых 3000 + длина предложений, результат = класс
            'reading_time (min)':          textstat.reading_time(text), #приблизительное время чтения в минутах (по длине текста)
        })

        # ----- ruts -----
        rs = ReadabilityStats(text)
        ds = DiversityStats(text)

        stats = rs.get_stats() # метрики удобочитаемости 
        div  = ds.get_stats() # лексическое разнообразие

        ruts_list.append({
            'Name': name,
            'Year': year,
            'Flesch Reading Ease':         stats['flesch_reading_easy'],
            'Flesch-Kincaid Grade':        stats['flesch_kincaid_grade'],
            'SMOG':                         stats['smog_index'],
            'Coleman-Liau Index':          stats['coleman_liau_index'],
            'Automated Readability Index': stats['automated_readability_index'],
            'LIX':                          stats['lix'], # слов/предложение + % «длинных» слов, Результаты: <30 – очень простой (детская литература), 40–50 – средний (например, журнальные статьи), >60 – чрезвычайно трудный текст (официально-деловой стиль, законы)
            'TTR':                          div['ttr'], #Type-Token Ratio, мера лексического разнообразия (уникальных слов)
        })

    # Превращаем в DataFrame
    df_textstat = pd.DataFrame(ts_list)
    df_ruts = pd.DataFrame(ruts_list)

In [19]:
# Вывод
print("\n=== Метрики textstat ===")
print(df_textstat.to_string(index=False))

print("\n=== Метрики ruts ===")
print(df_ruts.to_string(index=False))


=== Метрики textstat ===
                           Name Year  Flesch Reading Ease  Flesch-Kincaid Grade     SMOG  Coleman-Liau Index  Automated Readability Index  Gunning Fog  Dale-Chall Score  reading_time (min)
MitrofanovaGolubev_IMS_2024_rus 2024            99.964223              4.026105 5.502600           23.171303                    22.622162     7.947541         20.342045           343.76069
            Sukhan_IMS_2024_rus 2024           103.194765              3.006552 4.927825           22.059736                    20.153757     6.905152         20.216489           401.63929
          Khodorkovsky_IMS_2023 2023           103.275802              3.494815 3.129100           22.319586                    21.092203     7.471605         20.328628           476.69050
         Vybornaya_IMS_2024_rus 2024           105.219193              2.588988 4.059921           22.114270                    20.666996     6.515419         20.217579           193.40854
         ChizhikEgorov_IMS_20

In [21]:
output_path = '/Users/juliak/Downloads/readability_metrics.xlsx'  # или любой другой путь

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_textstat.to_excel(writer, sheet_name='textstat', index=False)
    df_ruts.to_excel(writer,    sheet_name='ruts',     index=False)

Выводы: 
По «русским» метрикам тексты имеют высокий лексико-синтаксический уровень (требуют подготовки ~10–20 лет обучения, LIX >80 относятся к очень сложным).

По «английским» метрикам те же тексты оценены как простые из-за использования других коэффициентов.